# Phase 1 — Causal Unit and Temporal Framework Audit

This notebook is the updated Phase 1 audit. It is intentionally limited to the causal unit, timestamp validation, repeated-ECG handling, index-ECG candidate comparison, and time-zero suitability.

No treatment analysis, propensity scoring, DAG construction, or causal discovery is performed here.


In [14]:
import json
from pathlib import Path

import pandas as pd

base = Path(r'c:/Users/anura/OneDrive/Desktop/capstone2/master')
available = {split: base / f'{split}.parquet' for split in ['train', 'validation', 'test']}
available = {split: path for split, path in available.items() if path.exists()}
print('available split files:', list(available.keys()))
for split, path in available.items():
    df = pd.read_parquet(path)
    print(f'{split} rows = {len(df)}')
if available:
    combined = pd.concat([pd.read_parquet(path) for path in available.values()], ignore_index=True)
    print('combined rows =', len(combined))
else:
    print('No split parquet files were found.')


available split files: ['train', 'validation', 'test']
train rows = 262813
validation rows = 56319
test rows = 56798
combined rows = 375930


## 1. Verify the split structure

The actual parquet files present in the `master/` directory were checked directly. The split rows below are therefore based on the files that actually exist in the workspace, not on assumed split metadata.

Reported split counts:

- train rows
- validation rows
- test rows
- combined rows

If separate validation/test files are available, the split-overlap audit for `SubjectID`, `hadm_id`, and `study_id` is performed; otherwise, it is recorded as unresolved.


In [15]:
split_rows = {split: 0 for split in ['train', 'validation', 'test']}
for split, path in available.items():
    split_rows[split] = len(pd.read_parquet(path))
combined = pd.concat([pd.read_parquet(path) for path in available.values()], ignore_index=True) if available else pd.DataFrame()
print('train rows =', split_rows.get('train', 0))
print('validation rows =', split_rows.get('validation', 0))
print('test rows =', split_rows.get('test', 0))
print('combined rows =', len(combined))
if len(available) >= 3:
    keys = ['SubjectID', 'hadm_id', 'study_id']
    for key in keys:
        vals = [set(pd.read_parquet(path)[key].dropna().tolist()) for path in available.values()]
        print(f'{key} overlap across all available splits = {len(set.intersection(*vals))}')
else:
    print('split overlap audit: unresolved because all split files are not available')


train rows = 262813
validation rows = 56319
test rows = 56798
combined rows = 375930
SubjectID overlap across all available splits = 0
hadm_id overlap across all available splits = 0
study_id overlap across all available splits = 0


### Split audit conclusion

The actual available data in this workspace are three separate parquet files with rows: train = 262,813; validation = 56,319; test = 56,798; combined = 375,930. Therefore the data are a combined master dataset composed of predefined train/validation/test parquet files, not a single 375,930-row parquet that was later split without preserving the split files.

The split-overlap audit for `SubjectID`, `hadm_id`, and `study_id` is performed only when all split files are available.


## 2. Verify `study_id` uniqueness

The expected interpretation is that `study_id` identifies each ECG observation. We simply report whether duplicates are present.


In [16]:
combined = pd.concat([pd.read_parquet(path) for path in available.values()], ignore_index=True) if available else pd.DataFrame()
study_summary = {
    'total_ecg_rows': int(len(combined)),
    'unique_study_id': int(combined['study_id'].nunique()),
    'duplicate_study_id_count': int(combined['study_id'].duplicated().sum()),
}
print(json.dumps(study_summary, indent=2))


{
  "total_ecg_rows": 375930,
  "unique_study_id": 375930,
  "duplicate_study_id_count": 0
}


## 3. Check MI-event multiplicity

Because admission-level has been selected as the primary observational unit, we verify whether an admission maps to a single `index_mi_time`.


In [17]:
mi_events = combined[combined['index_mi_time'].notna()].copy()
hadm_distinct = mi_events.groupby('hadm_id')['index_mi_time'].nunique()
subject_distinct = mi_events.groupby('SubjectID')['index_mi_time'].nunique()
print('number of distinct non-null index_mi_time values per hadm_id =', hadm_distinct.describe().to_dict())
print('number of admissions with >1 distinct index_mi_time =', int((hadm_distinct > 1).sum()))
print('number of distinct non-null index_mi_time values per SubjectID =', subject_distinct.describe().to_dict())
print('number of patients with >1 distinct index_mi_time =', int((subject_distinct > 1).sum()))


number of distinct non-null index_mi_time values per hadm_id = {'count': 14559.0, 'mean': 1.0, 'std': 0.0, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 1.0, 'max': 1.0}
number of admissions with >1 distinct index_mi_time = 0
number of distinct non-null index_mi_time values per SubjectID = {'count': 11840.0, 'mean': 1.2296452702702703, 'std': 0.706335555691594, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 1.0, 'max': 14.0}
number of patients with >1 distinct index_mi_time = 1776


## 4. Compare Candidate B vs Candidate C selections

The earlier audit reported identical counts for B and C, but counts alone do not imply that the same ECG is selected. We compare the actual selected `study_id` for B and C within the same admission.


In [18]:
valid = combined[combined['index_mi_time'].notna()].copy()
valid['ecg_time_dt'] = pd.to_datetime(valid['ecg_time'], errors='coerce', utc=True)
valid['index_mi_time_dt'] = pd.to_datetime(valid['index_mi_time'], errors='coerce', utc=True)
before = valid[valid['ecg_time_dt'] < valid['index_mi_time_dt']].copy().sort_values(['SubjectID', 'hadm_id', 'ecg_time_dt'])
B = before.drop_duplicates(subset=['SubjectID', 'hadm_id'], keep='first').set_index(['SubjectID', 'hadm_id'])
before_c = before.copy()
before_c['distance_to_mi_hours'] = (before_c['index_mi_time_dt'] - before_c['ecg_time_dt']).dt.total_seconds() / 3600.0
C = before_c.sort_values(['SubjectID', 'hadm_id', 'distance_to_mi_hours', 'ecg_time_dt']).drop_duplicates(subset=['SubjectID', 'hadm_id'], keep='first').set_index(['SubjectID', 'hadm_id'])
common = B.index.intersection(C.index)
B_sel = B.loc[common, 'study_id']
C_sel = C.loc[common, 'study_id']
different = int((B_sel.to_numpy() != C_sel.to_numpy()).sum())
print('number of admissions where B and C select different ECGs =', different)
print('percentage of eligible admissions where they differ =', float(different / len(common)) if len(common) else 0.0)
if different == 0:
    print('B and C select the same ECG in all eligible admissions.')
else:
    print('B and C differ on a non-trivial fraction of eligible admissions; identical counts do not imply identical selections.')


number of admissions where B and C select different ECGs = 5913
percentage of eligible admissions where they differ = 0.505082429315794
B and C differ on a non-trivial fraction of eligible admissions; identical counts do not imply identical selections.


## 5. Audit ECG timestamps and MI-event timestamps

We verify whether `time_diff_hours` matches `ecg_time - index_mi_time` in hours and report summary statistics on the valid pairs.


In [19]:
combined['ecg_time_dt'] = pd.to_datetime(combined['ecg_time'], errors='coerce', utc=True)
combined['index_mi_time_dt'] = pd.to_datetime(combined['index_mi_time'], errors='coerce', utc=True)
mask = combined['ecg_time_dt'].notna() & combined['index_mi_time_dt'].notna()
actual_hours = (combined.loc[mask, 'ecg_time_dt'] - combined.loc[mask, 'index_mi_time_dt']).dt.total_seconds() / 3600.0
reported = pd.to_numeric(combined.loc[mask, 'time_diff_hours'], errors='coerce')
diff = actual_hours - reported
summary = {
    'n_valid_pairs': int(mask.sum()),
    'time_diff_min': float(reported.min()),
    'time_diff_max': float(reported.max()),
    'time_diff_mean': float(reported.mean()),
    'time_diff_median': float(reported.median()),
    'time_diff_std': float(reported.std(ddof=1)),
    'n_positive': int((reported > 0).sum()),
    'n_negative': int((reported < 0).sum()),
    'n_approx_zero': int((reported.abs() < 1e-9).sum()),
    'max_abs_error_hours': float(diff.abs().max()),
    'fraction_exact_match': float((diff.abs() < 1e-9).mean()),
}
print(json.dumps(summary, indent=2))


{
  "n_valid_pairs": 48230,
  "time_diff_min": -4599.716666666666,
  "time_diff_max": 5637.633333333333,
  "time_diff_mean": 43.19509053839243,
  "time_diff_median": 0.1666666666666666,
  "time_diff_std": 242.97860501970587,
  "n_positive": 24531,
  "n_negative": 23658,
  "n_approx_zero": 41,
  "max_abs_error_hours": 9.094947017729282e-13,
  "fraction_exact_match": 1.0
}


### Conclusion on sign convention

The arithmetic matches exactly: `time_diff_hours` = `ecg_time - index_mi_time` in hours. There is no sign inversion. Positive values mean the ECG occurred after the MI event; negative values mean before the MI event.


## 6. Verify temporal ordering

Using the actual timestamps, we classify valid MI-related rows as occurring before, approximately at, or after the MI event. The ±2-hour window is an operational temporal classification inherited from the project's existing MI temporal taxonomy. It is not being presented as an independently established causal threshold.


In [20]:
valid = combined[combined['index_mi_time_dt'].notna()].copy()
valid['actual_diff_hours'] = (valid['ecg_time_dt'] - valid['index_mi_time_dt']).dt.total_seconds() / 3600.0
valid['temporal_relation'] = 'missing_index_time'
valid.loc[valid['actual_diff_hours'] < -2, 'temporal_relation'] = 'ECG before MI'
valid.loc[(valid['actual_diff_hours'] >= -2) & (valid['actual_diff_hours'] <= 2), 'temporal_relation'] = 'ECG approximately at MI'
valid.loc[valid['actual_diff_hours'] > 2, 'temporal_relation'] = 'ECG after MI'
print(valid['temporal_relation'].value_counts().to_string())
print('Cross-tab with Label_Troponin:')
print(pd.crosstab(valid['Label_Troponin'], valid['temporal_relation']).to_string())


temporal_relation
ECG after MI               21917
ECG before MI              20287
ECG approximately at MI     6026
Cross-tab with Label_Troponin:
temporal_relation      ECG after MI  ECG approximately at MI  ECG before MI
Label_Troponin                                                             
Control_Symptomatic           21917                     3428          12382
MI_Acute_Presentation             0                     2598           1953
MI_Pre-Incident                   0                        0           5952


### Label/timestamp wording

**The label taxonomy and timestamp-derived temporal relationship encode different information and cannot be used interchangeably. Their discordance requires explicit handling in the causal cohort definition.**

The existing cross-tabulation above is preserved; it shows the label set and the timestamp-derived temporal relationship do not map one-to-one.


### UTC wording

**Timestamps were parsed as timezone-aware UTC timestamps for consistent arithmetic. The source strings do not contain explicit timezone offsets, so UTC is an analysis convention rather than an independently verified source timezone.**


## Decision Record

- Admission-level is the primary observational unit.
- ECG-level observations are not treated as independent causal units because repeated ECGs are common within admissions and multiple admissions are common within patients.
- `study_id` is unique in the current data and therefore identifies ECG observations without duplication.
- The timestamp arithmetic is internally consistent: `time_diff_hours` matches `ecg_time - index_mi_time` in hours.
- The ±2-hour classification is an operational temporal taxonomy inherited from the project and is not an independently established causal threshold.
- ECG time is not yet established as causal time zero.
- Index-ECG selection remains unresolved pending the Phase 2 treatment-timing audit.

**Phase 1 status: unresolved pending Phase 2 treatment-timing audit**

**Handoff to Phase 2:** Phase 2 should audit treatment timing, exposure assignment timing, and the clinically defensible time-zero definition before any causal treatment analysis is attempted.
